# Automatic curation, and export for manual curation (Spyglass pipeline, step 2 of 4)

This notebook continues the Spyglass spike-sorting pipeline from
[`Pipeline_Spyglass_SpikeSorting.ipynb`](Pipeline_Spyglass_SpikeSorting.ipynb), which sorted the
recording and registered the raw sorter output as `CurationV1` **`curation_id = 0`**. Here we
*curate* that sort two different ways:

1. **Automatic curation** — Spyglass's native `MetricCuration`: compute quality metrics and apply
   threshold-based labels, stored as a new curation (`curation_id = 1`).
2. **Manual curation** — we *export* the recording and sorting so they can be hand-curated in the
   SpikeInterface GUI. The GUI cannot run in this environment, so that step happens in a separate
   notebook ([`Pipeline_Spyglass_ManualCuration.ipynb`](Pipeline_Spyglass_ManualCuration.ipynb))
   run with the **`spikeinterface_gui_env`** kernel.

Step 4 ([`Pipeline_Spyglass_CompareCurations.ipynb`](Pipeline_Spyglass_CompareCurations.ipynb))
re-ingests the manual result and compares all three curation rounds. See
[`README.md`](README.md) for the whole chain.

> **Environment:** run this notebook with the **`spyglass`** kernel (SpikeInterface 0.99).

## Connect to the database

In [ ]:
import json
from pathlib import Path
from pprint import pprint

import datajoint as dj
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spikeinterface as si

# Load config for database connection info
dj_local_conf_path = "/Users/pauladkisson/Documents/CatalystNeuro/Spyglass/spyglass/dj_local_conf.json"
dj.config.load(dj_local_conf_path)

# Spyglass stores some parameters as native Python objects (dicts / lists) in the database;
# this flag lets DataJoint serialize and deserialize those blobs instead of rejecting them.
dj.config["enable_python_native_blobs"] = True

# General Spyglass imports (importing common connects to the database)
import spyglass.common as sgc
import spyglass.spikesorting.v1 as sgs
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename

## Parameters set manually

These must match the values used in the sorting notebook.

In [ ]:
### Parameters set manually ###

# Session: the NWB copy that lives in the database (note the trailing underscore)
nwb_file_name = get_nwb_copy_filename("H3022-210806.nwb")  # -> "H3022-210806_.nwb"

# Which shank (sort group) and which epoch (interval) were sorted
sort_group_id = 0
interval_list_name = "01"  # first wake epoch for this session

# Pipeline parameters (must match the values used in Pipeline_Spyglass_SpikeSorting.ipynb)
preproc_param_name = "default"
sorter = "mountainsort5"
sorter_param_name = "default"

# Shared handoff folder for the manual-curation step (notebooks 2 -> 3 -> 4)
export_root = Path("manual_curation_export")

## Recover the raw sort

Find the sorting from step 1 and confirm its raw curation exists.

In [ ]:
# Re-derive the sorting produced by Pipeline_Spyglass_SpikeSorting.ipynb from the parameters
# above. Every stage is keyed off the recording, so we first recover its recording_id, then the
# sorting_id, then point at the raw curation (curation_id = 0) that notebook registered.
recording_id = (
    sgs.SpikeSortingRecordingSelection
    & {
        "nwb_file_name": nwb_file_name,
        "sort_group_id": sort_group_id,
        "interval_list_name": interval_list_name,
        "preproc_param_name": preproc_param_name,
    }
).fetch1("recording_id")

sorting_id = (
    sgs.SpikeSortingSelection
    & {
        "recording_id": recording_id,
        "sorter": sorter,
        "sorter_param_name": sorter_param_name,
    }
).fetch1("sorting_id")

curation_key = {"sorting_id": sorting_id, "curation_id": 0}
assert sgs.CurationV1 & curation_key, (
    "No raw curation (curation_id = 0) found for this sorting. "
    "Run Pipeline_Spyglass_SpikeSorting.ipynb first."
)
sgs.CurationV1 & {"sorting_id": sorting_id}

## 1. Automatic curation with quality metrics

`MetricCuration` extracts waveforms, computes quality metrics (SNR, ISI violations, isolation,
noise overlap, ...), and then applies **label rules** to flag low-quality units. The result is
inserted as a new `CurationV1` row whose parent is the raw sort (`curation_id = 0`).

**A pitfall to know about.** The shipped `"default"` metric-curation params label a unit only when
`nn_noise_overlap > 0.1`. On this example recording no unit exceeds that threshold, so *every*
label list comes back empty — and `MetricCuration.populate` then crashes while writing an
all-empty `curation_label` column (`could not resolve dtype ... empty list`). To get a working,
illustrative example we register our own params below that threshold SNR and ISI violations, which
reliably flag the weakest mountainsort5 clusters.

> ⚠️ **Tune these thresholds for your own data.** After populating, the cell below prints how many
> units were labeled. If that count is **0**, relax the thresholds (e.g. raise the SNR cutoff)
> until at least one unit is labeled, otherwise the populate step will crash as described above.

In [ ]:
# Make sure the upstream parameter sets MetricCuration depends on exist.
sgs.WaveformParameters.insert_default()
sgs.MetricParameters.insert_default()

# Custom label rules of the form {metric: [operator, threshold, [labels]]}. "snr" and
# "isi_violation" are both computed by the "franklab_default" metric set used below. Valid Spyglass
# labels are: reject, noise, artifact, mua, accept.
label_params = {
    "snr": ["<", 2.0, ["noise"]],
    "isi_violation": [">", 0.5, ["mua"]],
}
sgs.MetricCurationParameters.insert1(
    {
        "metric_curation_param_name": "wood_default",
        "label_params": label_params,
        "merge_params": {},
    },
    skip_duplicates=True,
)

In [ ]:
# Select the raw sort + the waveform / metric / label parameter sets, then compute the metrics.
metric_curation_key = {
    "sorting_id": sorting_id,
    "curation_id": 0,
    "waveform_param_name": "default_not_whitened",
    "metric_param_name": "franklab_default",
    "metric_curation_param_name": "wood_default",
}
selection = sgs.MetricCurationSelection.insert_selection(metric_curation_key)
mc_key = {"metric_curation_id": selection["metric_curation_id"]}
sgs.MetricCuration.populate(mc_key)

In [ ]:
# Pull the computed labels, merge groups, and metrics back out...
labels = sgs.MetricCuration.get_labels(mc_key)
merge_groups = sgs.MetricCuration.get_merge_groups(mc_key)
metrics = sgs.MetricCuration.get_metrics(mc_key)

n_units = len(metrics["snr"])
n_labeled = sum(1 for unit_labels in labels.values() if unit_labels)
print(f"{n_labeled} of {n_units} units received a label")
assert n_labeled > 0, (
    "No unit was labeled -- relax the thresholds in `label_params` above, or MetricCuration "
    "will crash on an all-empty curation_label column."
)
pprint(labels)

In [ ]:
# ...and store them as a new curation (curation_id = 1), branching off the raw sort.
sgs.CurationV1.insert_curation(
    sorting_id=sorting_id,
    parent_curation_id=0,
    labels=labels,
    merge_groups=merge_groups,
    metrics=metrics,
    description="after metric curation",
)
sgs.CurationV1 & {"sorting_id": sorting_id}

## 2. Export the sorting for manual curation

The SpikeInterface GUI needs a `SortingAnalyzer`, which only exists in newer SpikeInterface
(0.101+). This `spyglass` environment ships SpikeInterface 0.99, and the GUI is not installed here
at all — so manual curation must happen in the separate `spikeinterface_gui_env`.

To bridge the two environments we export the **raw** recording and sorting (`curation_id = 0`) to
portable on-disk folders. The next notebook loads them, builds an analyzer, and launches the GUI.
We export the raw sort (not the auto-curated one) so the human sees every unit.

In [ ]:
export_dir = export_root / str(sorting_id)
export_dir.mkdir(parents=True, exist_ok=True)

recording = sgs.CurationV1.get_recording(curation_key)  # filtered, referenced spike-band recording
sorting = sgs.CurationV1.get_sorting(curation_key)       # raw mountainsort5 sorting

# Binary recording folder + npz sorting folder are both portable across SpikeInterface versions.
recording.save(folder=export_dir / "recording", format="binary", overwrite=True)
sorting.save(folder=export_dir / "sorting", overwrite=True)

print("Exported recording + sorting to:")
print("   ", export_dir.resolve())
print("\nNext steps:")
print("  1. Open Pipeline_Spyglass_ManualCuration.ipynb with the 'spikeinterface_gui_env' kernel.")
print(f"  2. Set its `export_dir` to the path above (sorting_id = {sorting_id}).")
print("  3. Curate in the GUI, then return to Pipeline_Spyglass_CompareCurations.ipynb.")